<a href="https://colab.research.google.com/github/KelvinPogo/sdss_datathon_2026/blob/main/My_copy_SDSS_datathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install pgeocode

In [4]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
import pandas as pd
file_path = '/content/drive/MyDrive/public_services_dataset.xlsx'
df = pd.read_excel(file_path)
print(df.head())

  OCCUPANCY_DATE LOCATION_POSTAL_CODE       SECTOR OVERNIGHT_SERVICE_TYPE  \
0     2024-01-01               M9W1J1     Families    Motel/Hotel Shelter   
1     2024-01-01               M9W1J1  Mixed Adult    Motel/Hotel Shelter   
2     2024-01-01               M5S2P1  Mixed Adult                Shelter   
3     2024-01-01               M2J4R1     Families    Motel/Hotel Shelter   
4     2024-01-01               M2J4R1     Families    Motel/Hotel Shelter   

  PROGRAM_MODEL                PROGRAM_AREA        CAPACITY_TYPE  \
0     Emergency  Temporary Refugee Response  Room Based Capacity   
1     Emergency  Temporary Refugee Response  Room Based Capacity   
2     Emergency      Base Program - Refugee   Bed Based Capacity   
3     Emergency          Temporary Programs  Room Based Capacity   
4     Emergency  Temporary Refugee Response  Room Based Capacity   

   ACTUAL_CAPACITY  OCCUPIED_CAPACITY  UNAVAILABLE_CAPACITY  OCCUPANCY_RATE  
0              149                149             

In [ ]:
df[df["PROGRAM_MODEL"].isna()]

,OCCUPANCY_DATE,LOCATION_POSTAL_CODE,SECTOR,OVERNIGHT_SERVICE_TYPE,PROGRAM_MODEL,PROGRAM_AREA,CAPACITY_TYPE,ACTUAL_CAPACITY,OCCUPIED_CAPACITY,UNAVAILABLE_CAPACITY,OCCUPANCY_RATE
52524,2025-01-25,M4K3W5,Youth,Top Bunk Contingency Space,NaN,Winter Programs,Bed Based Capacity,5,1,0,0.2
52547,2025-01-26,M4K3W5,Youth,Top Bunk Contingency Space,NaN,Winter Programs,Bed Based Capacity,5,4,0,0.8
52841,2025-01-27,M4K3W5,Youth,Top Bunk Contingency Space,NaN,Winter Programs,Bed Based Capacity,5,5,0,1.0
52960,2025-01-28,M4K3W5,Youth,Top Bunk Contingency Space,NaN,Winter Programs,Bed Based Capacity,4,4,1,1.0


In [ ]:
df_clean = df.replace(r'^\s*$', pd.NA, regex=True)
rows_with_missing = df_clean[df_clean.isna().any(axis=1)]
print(len(rows_with_missing))

4


In [ ]:
df_new = df.drop(index=[52524, 52547, 52841, 52960])

In [ ]:
df_clean = df_new.replace(r'^\s*$', pd.NA, regex=True)
rows_with_missing = df_clean[df_clean.isna().any(axis=1)]
print(len(rows_with_missing))

0


In [ ]:
df_clean = df_new.replace(
    to_replace=r'^\s*(nan|NAN)\s*$',
    value=pd.NA,
    regex=True
)

In [ ]:
rows_with_missing = df_clean[df_clean.isna().any(axis=1)]
print(rows_with_missing)

       OCCUPANCY_DATE LOCATION_POSTAL_CODE       SECTOR  \
22         2024-01-01                 <NA>  Mixed Adult   
23         2024-01-01                 <NA>  Mixed Adult   
66         2024-01-01                 <NA>        Youth   
67         2024-01-01                 <NA>        Youth   
86         2024-01-01                 <NA>  Mixed Adult   
...               ...                  ...          ...   
100261     2025-12-31                 <NA>  Mixed Adult   
100262     2025-12-31                 <NA>        Women   
100266     2025-12-31                 <NA>        Youth   
100301     2025-12-31                 <NA>  Mixed Adult   
100324     2025-12-31                 <NA>        Youth   

            OVERNIGHT_SERVICE_TYPE PROGRAM_MODEL  \
22             Motel/Hotel Shelter     Emergency   
23         Isolation/Recovery Site     Emergency   
66                         Shelter     Emergency   
67                         Shelter  Transitional   
86                         Shel

In [ ]:
# Remove rows that contain at least one real NaN
df_clean = df_clean.dropna()
print("Remaining rows:", len(df_clean))

Remaining rows: 95821


In [ ]:
df_clean.to_csv("clean_data.csv")

# Logistic Regression Model

In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/clean_data.csv")

# Ensure numeric
df["OCCUPANCY_RATE"] = pd.to_numeric(df["OCCUPANCY_RATE"], errors="coerce")

# Drop any rows where occupancy rate is missing (safety)
df = df.dropna(subset=["OCCUPANCY_RATE"])

# Create critical threshold (95%)
df["CRITICAL_95"] = (df["OCCUPANCY_RATE"] >= 0.95).astype(int)

# Optional: also create extreme threshold (100%)
df["FULL_100"] = (df["OCCUPANCY_RATE"] >= 1.0).astype(int)

print("Share of critical days:", df["CRITICAL_95"].mean())
print("Share of full days:", df["FULL_100"].mean())

Share of critical days: 0.8825309692029931
Share of full days: 0.7874891725195938


In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

In [10]:
# Ensure date format
df["OCCUPANCY_DATE"] = pd.to_datetime(df["OCCUPANCY_DATE"], errors="coerce")

# Create time features
df["MONTH"] = df["OCCUPANCY_DATE"].dt.month
df["DAY_OF_WEEK"] = df["OCCUPANCY_DATE"].dt.dayofweek

# Select features
features = [
    "ACTUAL_CAPACITY",
    "UNAVAILABLE_CAPACITY",
    "MONTH",
    "DAY_OF_WEEK",
    "SECTOR",
    "OVERNIGHT_SERVICE_TYPE",
    "PROGRAM_MODEL"
]

X = df[features]
y = df["CRITICAL_95"]

# One-hot encode categoricals
X = pd.get_dummies(X, drop_first=True)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [13]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.53      0.03      0.06      2295
           1       0.88      1.00      0.94     16870

    accuracy                           0.88     19165
   macro avg       0.70      0.51      0.50     19165
weighted avg       0.84      0.88      0.83     19165

ROC-AUC: 0.8076938603933965


In [14]:
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [15]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.32      0.78      0.46      2295
           1       0.96      0.78      0.86     16870

    accuracy                           0.78     19165
   macro avg       0.64      0.78      0.66     19165
weighted avg       0.89      0.78      0.81     19165

ROC-AUC: 0.8072553668770412


In [16]:
!pip install xgboost

In [17]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    scale_pos_weight=(len(y_train[y_train==0]) / len(y_train[y_train==1])),
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)
y_prob = xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [01:31:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              precision    recall  f1-score   support

           0       0.53      0.93      0.67      2295
           1       0.99      0.89      0.93     16870

    accuracy                           0.89     19165
   macro avg       0.76      0.91      0.80     19165
weighted avg       0.93      0.89      0.90     19165

ROC-AUC: 0.9646766442861148
